In [1]:
import pandas as pd

In [4]:
members = pd.read_csv("members.csv")
books = pd.read_csv("books.csv")
loans = pd.read_csv("Loans.csv")
bookcopies = pd.read_csv("BookCopies.csv")

In [1]:
import pandas as pd
import re
from datetime import datetime

def load_csv(path):
    df = pd.read_csv(path, dtype=str)
    # drop accidental empty unnamed columns
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]
    if unnamed:
        print(f"Dropping unnamed columns from {path}: {unnamed}")
        df = df.drop(columns=unnamed)
    return df

def is_valid_email(email):
    if pd.isna(email):
        return False
    return bool(re.fullmatch(r"[^@]+@[^@]+\.[^@]+", email.strip()))

def is_valid_phone(phone):
    if pd.isna(phone):
        return False
    phone = str(phone).strip()
    return bool(re.fullmatch(r"\d{7,15}", phone))

def as_datetime(series, fmt=None):
    return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)

def report(df, name):
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("missing per column:\n", df.isna().sum())
    print("duplicate rows:", df.duplicated().sum())

members = load_csv("members.csv")
books = load_csv("books.csv")
loans = load_csv("Loans.csv")
bookcopies = load_csv("BookCopies.csv")
staff = load_csv("staff.csv")

report(members, "members")
report(books, "books")
report(loans, "loans")
report(bookcopies, "bookcopies")
report(staff, "staff")

# 1) Business rule / invalid value checks

print("\n--- Value rule checks ---")

def check_allowed(df, column, allowed, df_name):
    bad = df.loc[~df[column].isin(allowed) & df[column].notna(), column].unique()
    if len(bad):
        print(f"{df_name}.{column} invalid values:", bad)
    else:
        print(f"{df_name}.{column} all valid")

check_allowed(members, "Gender", ["Male", "Female"], "members")
check_allowed(members, "TierID", ["T0001", "T0002", "T0003", "T0004"], "members")
check_allowed(books, "MinTierID", ["T0001", "T0002", "T0003", "T0004"], "books")
check_allowed(loans, "LoanStatus", ["Returned", "Lost"], "loans")
check_allowed(bookcopies, "Status", ["Available", "Unavailable", "Lost", "Checked Out"], "bookcopies")
check_allowed(staff, "Gender", ["Male", "Female"], "staff")
check_allowed(staff, "EmploymentType", ["Full-time", "Part-time"], "staff")

# 2) Referential integrity checks

print("\n--- Referential integrity ---")
def ref_check(child, child_col, parent, parent_col, child_name, parent_name):
    missing = child.loc[~child[child_col].isin(parent[parent_col]), child_col].unique()
    if len(missing):
        print(f"{child_name}.{child_col} values not found in {parent_name}.{parent_col}:", missing[:20])
    else:
        print(f"{child_name}.{child_col} all found in {parent_name}.{parent_col}")

ref_check(loans, "MemberID", members, "MemberID", "loans", "members")
ref_check(loans, "BookID", books, "BookID", "loans", "books")
ref_check(loans, "StaffID", staff, "StaffID", "loans", "staff")
ref_check(loans, "BookID", bookcopies, "BookID", "loans", "bookcopies")
ref_check(bookcopies, "BookID", books, "BookID", "bookcopies", "books")

# 3) Key integrity and consistency

print("\n--- Key integrity / duplicates ---")

def key_duplicates(df, cols, name):
    if any(col not in df.columns for col in cols):
        return
    dup = df[df.duplicated(subset=cols, keep=False)].sort_values(cols)
    print(f"{name} duplicates on {cols}: {len(dup)} rows")
    if not dup.empty:
        print(dup.head(10).to_string(index=False))

key_duplicates(members, ["MemberID"], "members")
key_duplicates(books, ["BookID"], "books")
key_duplicates(staff, ["StaffID"], "staff")
key_duplicates(bookcopies, ["BookID", "CopySeqNo"], "bookcopies")
key_duplicates(loans, ["MemberID", "BookID", "CopySeqNo", "LoanDatetime"], "loans")

# 4) Date validity checks

print("\n--- Date checks ---")
members["DateOfBirth_dt"] = as_datetime(members["DateOfBirth"])
members["RegistrationDatetime_dt"] = as_datetime(members["RegistrationDatetime"])
loans["LoanDatetime_dt"] = as_datetime(loans["LoanDatetime"])
loans["DueDate_dt"] = as_datetime(loans["DueDate"])
loans["ReturnDatetime_dt"] = as_datetime(loans["ReturnDatetime"])
staff["StartDatetime_dt"] = as_datetime(staff["StartDatetime"])
staff["DOB_dt"] = as_datetime(staff["DOB"])

print("members DateOfBirth parse failures:", members["DateOfBirth_dt"].isna().sum())
print("members RegistrationDatetime parse failures:", members["RegistrationDatetime_dt"].isna().sum())
print("loans LoanDatetime parse failures:", loans["LoanDatetime_dt"].isna().sum())
print("loans DueDate parse failures:", loans["DueDate_dt"].isna().sum())
print("loans ReturnDatetime parse failures:", loans["ReturnDatetime_dt"].isna().sum())
print("staff StartDatetime parse failures:", staff["StartDatetime_dt"].isna().sum())
print("staff DOB parse failures:", staff["DOB_dt"].isna().sum())

bad_dob = members.loc[members["DateOfBirth_dt"] > pd.Timestamp.today(), ["MemberID", "DateOfBirth"]]
if not bad_dob.empty:
    print("members with future DateOfBirth:")
    print(bad_dob.to_string(index=False))

bad_register = members.loc[
    (members["RegistrationDatetime_dt"].notna()) &
    (members["DateOfBirth_dt"].notna()) &
    (members["RegistrationDatetime_dt"] < members["DateOfBirth_dt"]),
    ["MemberID", "DateOfBirth", "RegistrationDatetime"]
]
if not bad_register.empty:
    print("members registered before DateOfBirth:")
    print(bad_register.to_string(index=False))

bad_due = loans.loc[
    loans["LoanDatetime_dt"].notna() &
    loans["DueDate_dt"].notna() &
    (loans["DueDate_dt"] < loans["LoanDatetime_dt"]),
    ["MemberID", "BookID", "LoanDatetime", "DueDate"]
]
if not bad_due.empty:
    print("loans with DueDate before LoanDatetime:")
    print(bad_due.head(20).to_string(index=False))

bad_return = loans.loc[
    loans["LoanDatetime_dt"].notna() &
    loans["ReturnDatetime_dt"].notna() &
    (loans["ReturnDatetime_dt"] < loans["LoanDatetime_dt"]),
    ["MemberID", "BookID", "LoanDatetime", "ReturnDatetime"]
]
if not bad_return.empty:
    print("loans with ReturnDatetime before LoanDatetime:")
    print(bad_return.head(20).to_string(index=False))

# 5) Format / completeness checks

print("\n--- Format and completeness checks ---")
members_email_bad = members.loc[~members["Email"].apply(is_valid_email)]
print("members bad emails:", len(members_email_bad))
members_phone_bad = members.loc[~members["ContactNo"].apply(is_valid_phone)]
print("members bad ContactNo:", len(members_phone_bad))
staff_email_bad = staff.loc[~staff["Contact"].apply(is_valid_email)]
print("staff bad Contact:", len(staff_email_bad))

for df, name in [(members, "members"), (books, "books"),
                 (loans, "loans"), (bookcopies, "bookcopies"), (staff, "staff")]:
    print(f"{name} missing rows count:", df.isna().any(axis=1).sum())

# 6) Check inconsistent categorical values

print("\n--- Categorical consistency ---")
for df, col, name in [
    (bookcopies, "Status", "bookcopies"),
    (members, "TierID", "members"),
    (books, "MinTierID", "books"),
    (staff, "EmploymentType", "staff")
]:
    if col in df.columns:
        vals = sorted(df[col].dropna().unique())
        print(f"{name}.{col} values:", vals)

Dropping unnamed columns from Loans.csv: ['Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

=== members ===
shape: (301, 9)
missing per column:
 MemberID                 0
FullName                 0
ContactNo                0
Email                    0
DateOfBirth              0
Gender                   0
Occupation              43
RegistrationDatetime     0
TierID                   0
dtype: int64
duplicate rows: 0

=== books ===
shape: (500, 5)
missing per column:
 BookID       0
Title        0
Author       0
Genre        0
MinTierID    0
dtype: int64
duplicate rows: 0

=== loans ===
shape: (12345, 8)
missing per column:
 MemberID          0
BookID            0
CopySeqNo         0
StaffID           0
LoanDatetime      0
DueDate           0
ReturnDatetime    1
LoanStatus        7
dtype: int64
duplicate rows: 0

=== bookcopies ===
shape: (1265, 3)
missing per column:
 BookID       0
CopySeqNo    0
Status       0
dtype: int64
duplicate rows: 0

=== 

C:\Users\JunHao\AppData\Local\Temp\ipykernel_52284\1466630940.py:26: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
C:\Users\JunHao\AppData\Local\Temp\ipykernel_52284\1466630940.py:26: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
C:\Users\JunHao\AppData\Local\Temp\ipykernel_52284\1466630940.py:26: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict